# 03 Feature engineering
Reads data.csv, builds the daily table and the features, saves data_stats.csv.
Anything from pollutants is shifted by at least one day. Weather is from the same day.

In [43]:
import pandas as pd
import numpy as np

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
BASE = '/content/drive/MyDrive/ml/weather'

## Daily aggregation
Days with fewer than 18 hourly PM10 values get NaN.

In [46]:
data = pd.read_csv(f'{BASE}/data.csv', index_col=0, parse_dates=True)
data_stats = data.resample('D').agg({
    'temp': ['mean', 'min', 'max'],
    'humidity':'mean',
    'precipitation':'sum',
    'pressure':'mean',
    'wind_speed': ['mean', 'max'],
    'wind_gusts':'max',
    'cloud_cover':'mean',
    'pm10':['mean', 'count']})

data_stats.columns = ['_'.join(col) for col in data_stats.columns]

In [47]:
cond = data_stats['pm10_count'] < 18
data_stats.loc[cond, 'pm10_mean'] = np.nan
data_stats = data_stats.drop('pm10_count', axis=1)

## Final features
Used by the model: the 16 base features + temp_dif, 3days_max, U_wind.

In [48]:
data_stats['pm10_lag1'] = data_stats['pm10_mean'].shift(1)
data_stats['pm10_lag7'] = data_stats['pm10_mean'].shift(7)
data_stats['pm10_roll7'] = data_stats['pm10_mean'].shift(1).rolling(7, min_periods=4).mean()
data_stats['3days_max'] = data_stats['pm10_mean'].shift(1).rolling(3).max()

In [49]:
data_stats['dayofyear'] = data_stats.index.dayofyear

In [50]:
# sin/cos for each hour first, then daily mean
wind_rad = np.radians(data['wind_dir']).rename('wind_rad')
wind_sin = np.sin(wind_rad).resample('D').mean().rename('wind_sin')
wind_cos = np.cos(wind_rad).resample('D').mean().rename('wind_cos')
data_stats = data_stats.join([wind_sin, wind_cos])

In [51]:
# daily temperature range
data_stats['temp_dif'] = data['temp'].resample('D').max() - data['temp'].resample('D').min()
# east-west wind component
data_stats['U_wind'] = data_stats['wind_speed_mean'] * data_stats['wind_sin']

## Candidate features
Tested in 04, not used in the final model.

In [52]:
# trend features
data_stats['pm10_lag2'] = data_stats['pm10_mean'].shift(2)
data_stats['pm10_diff'] = data_stats['pm10_lag1'] - data_stats['pm10_lag2']
data_stats['3days_std'] = data_stats['pm10_mean'].shift(1).rolling(3).std()

# calendar
data_stats['dayofweek'] = data_stats.index.dayofweek

In [53]:
# other pollutants: daily mean, then shift
data_stats['so2_lag1'] = data['so2'].resample('D').mean().shift(1)
data_stats['no2_lag1'] = data['no2'].resample('D').mean().shift(1)
# PM2.5 / PM10 ratio, previous day
data_stats['pollution'] = (data['pm2.5'] / data['pm10']).resample('D').mean().shift(1)

In [54]:
# wind speed is in km/h, so < 2 means almost calm
data_stats['low_wind_speed'] = (data['wind_speed'] < 2).resample('D').sum()
night_cond = data.index.hour.isin([21, 22, 23, 0, 1, 2, 3, 4, 5])
data_stats['night_wind_speed'] = data.loc[night_cond, 'wind_speed'].resample('D').mean()
data_stats['pressure_dif'] = data_stats['pressure_mean'] - data_stats['pressure_mean'].shift(1)
data_stats['V_wind'] = data_stats['wind_speed_mean'] * data_stats['wind_cos']

# heating season: 15 Oct - 15 Apr
heating_cond = (data_stats.index.month.isin([11, 12, 1, 2, 3])) | ((data_stats.index.month==10) & (data_stats.index.day > 14)) | ((data_stats.index.month==4) & (data_stats.index.day < 16))
data_stats['heating_season'] = heating_cond.astype(int)

## Save

In [55]:
print(data_stats.shape)
data_stats.to_csv(f'{BASE}/data_stats.csv')

(2217, 32)
